In [1]:
import pandas as pd
df = pd.read_csv("churn_data.csv")
df.head()

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [2]:
df.shape

(500, 6)

In [3]:
df.describe()

,age,salaire,anciennete,consommation,churn
count,500.000000,500.00000,500.000000,500.000000,500.000000
mean,44.220000,4666.30800,7.144000,48.660000,0.038000
std,15.036082,1844.86557,4.364558,28.927115,0.191388
min,18.000000,1209.00000,0.000000,0.000000,0.000000
25%,32.000000,3124.75000,3.000000,23.750000,0.000000
50%,45.000000,4679.00000,7.000000,47.000000,0.000000
75%,57.000000,6225.25000,11.000000,74.250000,0.000000
max,69.000000,7988.00000,14.000000,99.000000,1.000000


In [4]:
df.isnull().sum()

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [5]:
df["churn"].value_counts()

churn
0    481
1     19
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

# 1. Conversion de la colonne « abonnement » en nombres
df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2
})

# 2. Variables explicatives (features)
X = df[
    [
        "age",
        "salaire",
        "abonnement",
        "anciennete",
        "consommation"
    ]
]

# 3. Variable cible (target)
y = df["churn"]

# 4. Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,       # 20% des données réservées au test
    random_state=42      # même logique de reproductibilité que pour le dataset
)

In [7]:
X_train.shape

(400, 5)

In [8]:
X_test.shape

(100, 5)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Une forêt de 100 arbres de décision qui votent ensemble.
# max_depth=5 limite la complexité de chaque arbre.
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# .fit() déclenche l'APPRENTISSAGE : le modèle analyse X_train et y_train
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [10]:
# Prédictions sur les données de test (jamais vues pendant l'entraînement)
y_pred = model.predict(X_test)

# Pourcentage de bonnes réponses
print("Accuracy :", accuracy_score(y_test, y_pred))

# Précision et rappel, classe par classe
print(classification_report(y_test, y_pred))

Accuracy : 0.99
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



In [13]:
import joblib

# joblib.dump() sérialise (transforme en fichier binaire) le modèle entraîné
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [14]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
    n_estimators=50,    # moins d'arbres qu'à l'essai 1 (100 -> 50)
    max_depth=3,        # arbres moins profonds qu'à l'essai 1 (5 -> 3)
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))

joblib.dump(model, "model.pkl")   # écrase le fichier de l'Essai 1
print("model.pkl écrasé avec l'Essai 2")

Accuracy Essai 2 : 0.99
model.pkl écrasé avec l'Essai 2


In [15]:
loaded = joblib.load("model.pkl")

# 5 -> Essai 1 ou 2 ; 4 -> forcément l'Essai 3 (sans salaire)
print("Nombre de features attendu :", loaded.n_features_in_)

Nombre de features attendu : 5


In [16]:
# ---------- ESSAI 3 : on retire la colonne "salaire" ----------
X = df[["age", "abonnement", "anciennete", "consommation"]]   # salaire retiré volontairement
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 3 (sans salaire) :", accuracy_score(y_test, y_pred))

joblib.dump(model, "model.pkl")   # écrase le fichier de l'Essai 2
print("model.pkl écrasé avec l'Essai 3")

Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3


In [17]:
loaded = joblib.load("model.pkl")

# 5 -> Essai 1 ou 2 ; 4 -> forcément l'Essai 3 (sans salaire)
print("Nombre de features attendu :", loaded.n_features_in_)

Nombre de features attendu : 4


Question 1: Les accuracies obtenues sont 0.99 (Essai 1), 0.99 (Essai 2) et 1.0 (Essai 3). L'Essai 3 (sans la colonne salaire) semble le meilleur, mais je ne peux pas le prouver avec certitude. Le jeu de test ne contient que 100 clients dont seulement 5 en churn : la différence entre 0.99 et 1.0 correspond à un seul client. De plus, le modèle n'a pas de vrai pouvoir prédictif, car la règle de churn est calculée à partir de consommation et anciennete, qui sont aussi dans les features (fuite de données).

Question 2 : Non. Le fichier model.pkl a été écrasé deux fois. Après l'Essai 2 il attendait 5 features, après l'Essai 3 il en attend 4, et le modèle de l'Essai 1 n'existe plus nulle part. Je pourrais essayer de le réentraîner à la main si je me souviens des paramètres (n_estimators=100, max_depth=5, 5 features, random_state=42), mais ce ne serait pas une vraie récupération : rien ne prouve que j'obtienne exactement le même modèle, et sans historique je ne peux pas le vérifier. C'est le problème du « modèle champion écrasé » : il faudrait versionner les modèles (MLflow).

Question 3 : Non, ce n'est pas garanti. Même avec le même churn_data.csv et le même code, les résultats peuvent différer selon les versions des bibliothèques (ici pandas 3.0.6, scikit-learn 1.9.1, numpy 2.5.3), la version de Python (3.12), l'environnement virtuel ou le kernel choisi. Les graines aléatoires (seed=42, random_state=42) aident, mais elles ne suffisent pas à garantir la reproductibilité sur une autre machine. Le CSV lui-même peut aussi différer si quelqu'un régénère les données avec d'autres versions. C'est le problème du « ça marche chez moi » : il faudrait figer les versions (requirements.txt), versionner le code et les données (Git) et suivre les expériences (MLflow).